# Quantum walks

$\renewcommand{\ket}[1]{\left|#1\right\rangle}\renewcommand{\bra}[1]{\left\langle #1\right|}\renewcommand{\braket}[2]{\left\langle #1 \middle| #2 \right\rangle}\renewcommand{\ketbra}[2]{\left|#1\right\rangle\!\left\langle #2\right|}$This notebook shows how to implement the Szegedy quantum walk operator, a unitary operator whose spectrum is tightly related to the transition matrix of the Markov chain. This is the main object of the quantum algorithm based on spectral filtering, either through a sequence of QPE steps or through QSVT-based polynomial transformations.

**Table of contents**

1. Szegedy quantum walk operator
2. Summary of the resources needed
3. Boltzmann coin
4. Proposal move unitaries
5. Reflection and Accept-Path unitaries
6. Comparison with Lemieux et al.

## Szegedy quantum walk operator

Consider an MCMC on $n$ Ising spins, so the configuration space has size $2^n$; its transition matrix $P_{yx}\in\mathbb{R}^{2^n\times 2^n}$ is induced by a trial-move matrix $T_{yx}\in\mathbb{R}^{2^n\times 2^n}$ and by the Metropolis-Hastings acceptance probability $A_{yx}$, with accepted off-diagonal moves weighted by $T_{yx}A_{yx}$ and the rejected probability placed on the diagonal.

The quantum walk circuit is an operator acting on the Hilbert space $\mathcal{H}_a \otimes \mathcal{H}_b \otimes \mathcal{H}_c$, where $\mathcal{H}_a$ and $\mathcal{H}_b$ are two copies of the system register holding the MCMC state, and $\mathcal{H}_c$ is an auxiliary register called the coin. The walk operator and the transition matrix $P$ have related spectrum, so that a filter on the walk eigenphases corresponds to a filter on the eigenvalues of the transition matrix; in this way, one can extract the stationary component of the walk. 

The walk operator has the form

$$
W = R_0 V^\dagger B^\dagger F B V.
$$

where

$$
V\ket{x}_a\ket{0}_b
=
\ket{x}_a
\sum_y \sqrt{T_{yx}}\ket{y}_b
$$

is the trial-move unitary. The quantity $T_{yx}$ is nonzero when the trial move from state $x$ to state $y$ is allowed with nonzero proposal probability.
$$
B\ket{x}_a\ket{y}_b\ket{0}_c
=
\ket{x}_a\ket{y}_b
\left(
\sqrt{A_{yx}}\ket{0\cdots 0}_c
+
\sqrt{1-A_{yx}}\ket{\bot_{xy}}_c
\right),
$$ 
is the Boltzmann coin, here $A_{yx}$ is the probability of accepting the trial move from $x$ to $y$, determined by the energy difference between the two states and the inverse temperature; and the state $\ket{\bot_{xy}}_c$ is orthogonal to $\ket{0\cdots 0}_c$.

$$
\left\{\begin{align*}
F\ket{x}_a\ket{y}_b\ket{0\cdots 0}_c
&=
\ket{y}_a\ket{x}_b\ket{0\cdots 0}_c, \\
F\ket{x}_a\ket{y}_b\ket{\bot_{xy}}_c
&=
\ket{x}_a\ket{y}_b\ket{\bot_{xy}}_c.
\end{align*}\right.
$$ 

is denoted as the accept-path swap unitary. $R_0$ acts as the identity on $\mathcal{H}_a$ and as a reflection about $\ket{0\cdots 0}_b\ket{0\cdots 0}_c$ on $\mathcal{H}_b \otimes \mathcal{H}_c$.

We expect three properties to hold.

1. $U = V^\dagger B^\dagger F B V$ is a block encoding of $X$, the discriminant matrix of the MCMC:

    $$
    (\mathbb{I}_a \otimes \bra{0}_b \otimes \bra{0}_c)
    U
    (\mathbb{I}_a \otimes \ket{0}_b \otimes \ket{0}_c)
    =
    X.
    $$

    For a reversible Markov chain with transition matrix $P$ and stationary distribution $\pi$, the discriminant has entries $X_{yx}=\sqrt{P_{xy}P_{yx}}$; $X$ is similar to $P$ and therefore they share the same spectrum. 
2. The stationary distribution is encoded in the zero-ancilla invariant subspace. In particular, detailed balance implies that the coherent stationary state $\ket{\sqrt{\pi}}=\sum_x \sqrt{\pi_x}\ket{x}$ is a $+1$ eigenvector of $X$: $X\ket{\sqrt{\pi}} = \ket{\sqrt{\pi}}$. Therefore, because $U$ block-encodes $X$, the full state $\ket{\sqrt{\pi}}_a\ket{0}_b\ket{0}_c$ is a $+1$ eigenstate of $U$ with no leakage outside the zero subspace.
3. If $\lambda \in \mathrm{spec}(X)$, then the corresponding walk eigenvalues are $\exp(\pm i \arccos(\lambda)) \in \mathrm{spec}(W)$.


## Summary of the resources needed

We are interested in estimating the non-Clifford gates, since Clifford gates can often be processed classically as shown in `4_quantum_models.ipynb`. The non-Clifford count is not the right metric, as the runtime depends on the critical path of the circuit; therefore we focus on the non-Clifford depth.

The non-Clifford gates here are only $T$ gates, rotations $R_z(\theta)$ with arbitrary angle $\theta$, and Toffoli gates.

We summarize the non-Clifford depth and qubit cost of the proposal, Boltzmann coin, reflection, and accept-path blocks.

Here 
* $n$ is the number of spins, $\beta$ is the inverse temperature,
* $\varepsilon$ is the implementation error in operator norm,
* $r$ is the number of Trotter steps in the QEMC proposal move,
* $\alpha=2\left(\sum_i |h_i|+\sum_{i<j}|J_{ij}|\right)$ is a loose upper bound on the energy difference that we use as a normalization factor. For our SK instances, $\alpha$ scales as $\alpha\simeq \frac{2}{\sqrt{\pi}}n^{3/2}$ (cf. `1_model_and_queries.ipynb`).

We use the temporary variables for logarithmic terms to improve the readability:
* $\ell_\varepsilon:=\log_2(\varepsilon^{-1})$.
* $\ell_n := \lceil \log_2(n) \rceil$. 
* $\ell_{2n}:=\left\lceil\log_2(2n)\right\rceil$.
* $S:=1+\log_2 n+\log_2(\varepsilon^{-1})$.
* $\beta_{\rm eff}:=\max\{\beta,1\}$.
* $m:=\max\left\{2,\log_2\!\left(\max\left\{1,\frac{\beta_{\rm eff}\alpha}{2\ln(\varepsilon^{-1})}\right\}\right)\right\}$.
* $\ell_S:=\left\lceil\log_2\!\left(1+\frac{7}{2}S\right)\right\rceil$.
* $\ell_m:=\log_2(m)$.

| Component | Non-Clifford depth | Total qubits | 
|---|---:|---:|
| Proposal with uniform move | $0$ | $2n$ |
| Proposal with local spin flip move | $13\ell_n+15$ | $2n$ | 
| Proposal with quantum-enhanced move, Trotterized with $r$ layers | $1+r(n+2)$ | $2n$ | 
| Boltzmann coin, hybrid arithmetic | $162\ell_\varepsilon\ell_S + 54\ell_S - 93\ell_\varepsilon + 24\ell_{2n} + 49S + 28\ell_m - 12$ | $2n+4n^2+21n^2S+2$ |
| Reflection, hybrid arithmetic | $14\left\lceil\log_2(n+7S+3)\right\rceil-13$ | $2n+7S+6$ |
| Accept-path unitary, hybrid arithmetic | $28\left\lceil\log_2(4+7S)\right\rceil-23$ | $3n+7S+4$ |

## Boltzmann coin

The Boltzmann coin is the most expensive portion of the walk circuit. It is in charge of calculating whether to accept the move and must implement the necessary arithmetic to determine that. There are a few different ways one can implement the Boltzmann coin; here we show three of them.

* **Fully-phase arithmetic**: this is the most space-efficient way. We define a Hamiltonian acting on $2n$ qubits containing on the diagonal the difference of energies between the two configurations. A polynomial transformation of the desired function gets us closer to the square root of the acceptance rule. This takes $O(n)$ space and $O(n^2)$ depth.

This implementation is detailed in `3.1_fully_phase_arithmetic.ipynb`.

* **Hybrid fixed point/phase arithmetic**: a portion of fixed point arithmetic is used to determine the energy difference. This takes $O(n^2)$ space and $O(\log_2 n)$ depth. Then, a polynomial transformation of a many-body Hamiltonian allows us to implement the square exponential in the acceptance rule. This takes $O(n)$ space and $O(\log_2 n)$ depth.

This implementation is detailed in `3.2_hybrid_arithmetic.ipynb`.

* **Fully-fixed point arithmetic**: the non-linear transformation is calculated in fixed point, too. We expect this to have comparable space and depth asymptotics to the hybrid arithmetic, albeit with worse prefactors, and therefore it has not been tested here.

## Proposal move unitaries

### Uniform move

The uniform proposal acts on two $n$-qubit registers $\ket{x}_a\ket{y}_b$, where register $a$ stores the current configuration and register $b$ is initialized to $\ket{0^n}$. The implementation applies Hadamard gates to all qubits of register $b$, mapping it to the uniform superposition over all bit strings.

No auxiliary qubits are required. Since the implementation only uses Hadamard gates, it contains no non-Clifford operations.

### Local move

A local move with Hamming weight $k$ acts on $\ket{x}_a\ket{0^n}_b$ by first applying the Dicke-state preparation routine to register $b$, producing the uniform superposition over all $n$-bit masks $z$ with Hamming weight $k$. It then applies bitwise CNOTs from register $a$ to register $b$, producing
$$
\ket{x}_a\ket{0^n}_b
\mapsto
\ket{x}_a\frac{1}{\sqrt{\binom nk}}\sum_{|z|=k}\ket{x\oplus z}_b.
$$
Here we mainly use $k=1$, corresponding to a uniformly random single-spin flip, although the implementation supports any $1\le k\le n$.

The Dicke-state preparation uses no auxiliary qubits. The implementation follows a split-and-merge structure:
- first, the state is initialized in unary form as $\ket{1^k0^{n-k}}$;
- then, WDB blocks distribute the Hamming weight across a balanced binary partition tree;
- finally, SCS blocks convert the local unary states on the leaves into local Dicke states.

The SCS block is the local unary-to-Dicke conversion used on the leaves of the partition tree. For a block with parameter $k$, it acts on $k+1$ qubits. In the implementation, its type-I block uses a controlled-$R_y$ gate, and its type-II blocks use controlled-controlled-$R_y$ gates.

The WDB block distributes a unary Hamming-weight register across two child blocks of the partition tree. It is the internal-node operation used before the final SCS leaf conversion. The implementation uses controlled-$R_y$, controlled-controlled-$R_y$, CNOT ladders, and a Fredkin staircase.

The full Dicke-state preparation combines WDB and SCS blocks through a balanced partition tree. Let $L=\lceil n/k\rceil$ be the number of leaves. Each leaf has size at most $k$, and each internal node combines two child blocks using a WDB. Since a binary tree with $L$ leaves has $L-1$ internal nodes, the total gate counts are bounded by the cost of all leaf SCS blocks plus the cost of all internal WDB blocks.

For depth, operations on disjoint subtrees can be parallelized. Therefore the depth scales with the height of the balanced partition tree, rather than with the total number of internal nodes. This gives a logarithmic-in-$L$ contribution from the WDB layers, followed by the local SCS cost on the leaves.

For generic $k$:

| Component | Non-Clifford depth | Total qubits |
|---|---:|---:|
| SCS | $2k$ | $k+1$ |
| WDB | $4k^2+9k$ | $n$ |
| Dicke state preparation | $\log_2\!\left(\left\lceil \frac{n}{k}\right\rceil\right)(4k^2+9k)+5k^2+10k$ | $n$ |

For $k=1$:

| Component | Non-Clifford depth | Total qubits |
|---|---:|---:|
| SCS | $2$ | $2$ |
| WDB | $13$ | $n$ |
| Dicke state preparation | $13\log_2(n)+15$ | $n$ |

### QEMC move

The QEMC proposal uses Hamiltonian evolution under a transverse-field Ising Hamiltonian to generate a non-local move. The proposal acts on two $n$-qubit registers $\ket{x}_a\ket{y}_b$.

The implementation first copies $x$ into the $b$-register using bitwise CNOTs. It then applies a second-order Trotter approximation of the transverse-field Ising evolution to register $b$. Since the copy layer is Clifford, the non-Clifford resources of the proposal come from the Hamiltonian simulation block.

The transverse-field Ising Hamiltonian is
$$
H=\sum_i h_i Z_i+\sum_{i<j}J_{ij}Z_iZ_j+\sum_i\gamma_iX_i.
$$
The implementation splits it into a commuting diagonal part $H_Z$ and a transverse-field part $H_X$, and applies the second-order Strang formula
$$
S_2(\Delta t)=e^{-i\Delta t H_X/2}e^{-i\Delta t H_Z}e^{-i\Delta t H_X/2}
$$
for a fixed number of Trotter steps $r$, with $\Delta t=t/r$.

The depth estimate exploits the grouped structure of the implementation. The $X_i$ rotations form a parallel single-qubit layer, the $Z_i$ rotations form another parallel single-qubit layer, and the $Z_iZ_j$ rotations are scheduled by a round-robin edge coloring of the complete graph. Therefore the ZZ part has depth at most $n$ matching rounds, even for dense all-to-all couplings.

For one Trotter step, this gives the coarse non-Clifford depth bound $n+2$ used in the resource table above, and the initial copy layer contributes one additional Clifford depth layer. Hence the proposal-depth estimate used here is $1+r(n+2)$, with $2n$ qubits.

## Reflection and Accept-Path unitaries

The reflection acts on $\mathcal H_A\otimes\mathcal H_B\otimes\mathcal H_{\rm coin}$ as
$$
R=2\left(I_A\otimes \ket{0^n}\!\bra{0^n}_B\otimes \ket{0^c}\!\bra{0^c}_{\rm coin}\right)-I,
$$
where $c$ is the number of coin qubits. The implementation maps the all-zero condition on $B\otimes{\rm coin}$ to an all-one condition using $X$ gates, applies a selective phase flip through an MCX conjugated by Hadamards on the target, and then uncomputes the $X$ gates. For more than two controls, the MCX uses the two-clean-ancilla construction `synth_mcx_2_clean_kg24` from [arXiv:2407.17966](https://arxiv.org/abs/2407.17966), with non-Clifford depth $14\lceil\log_2 q\rceil-13$ for $q$ controls; for one and two controls, the implementation uses `cx` and `ccx`.

The accept-path unitary swaps the two system registers only when the coin register is zero:
$$
F\ket{x}_A\ket{y}_B\ket{z}_{\rm coin}
=
\begin{cases}
\ket{y}_A\ket{x}_B\ket{z}_{\rm coin}, & z=0^c,\\
\ket{x}_A\ket{y}_B\ket{z}_{\rm coin}, & z\ne 0^c.
\end{cases}
$$
The implementation first computes the zero-coin flag using the same MCX construction, fans this flag out to $n$ controls using a logarithmic-depth CNOT fanout with $n-1$ ancillas, applies $n$ controlled swaps, and then uncomputes the fanout and flag. Each controlled swap is decomposed into three `ccx` gates, so the swap layer contributes non-Clifford depth $3$.

| Component | Non-Clifford depth | Total qubits |
|---|---:|---:|
| Reflection | $14\lceil\log_2(n+c-1)\rceil-13$ | $2n+c+2$ |
| Accept-path unitary | $28\lceil\log_2 c\rceil-23$ | $3n+c$ |

The coin-register size depends only on the arithmetic used.

For the hybrid arithmetic, the full Boltzmann block acts on $O(n^2\log n)$ qubits, but most of these are temporary auxiliary qubits that are uncomputed. The only non-uncomputed contribution is the one from `SqrtExpArithmetic`. This class acts on $3W+2$ qubits, where $W$ is the fixed-point width. However, the $W$-qubit signal register is not part of the persistent coin register: it is cleaned by the surrounding hybrid arithmetic. Therefore the effective coin register has $c=2W+2$ qubits.

In our case, $W=1+\frac{7}{2}S$, with $S:=1+\log_2 n+\log_2(\varepsilon^{-1})$. Hence $c=2W+2=4+7S$. Assuming $n\ge3$, this leads to the following table:

| Component | Non-Clifford depth | Total qubits |
|---|---:|---:|
| Reflection | $14\left\lceil\log_2(n+7S+3)\right\rceil-13$ | $2n+7S+6$ |
| Accept-path unitary | $28\left\lceil\log_2(4+7S)\right\rceil-23$ | $3n+7S+4$ |

## Comparison with Lemieux et al.

Our implementation uses a general arithmetic block for the Metropolis acceptance rule. This block computes the full energy difference associated with a proposed move and therefore does not rely on locality or sparsity assumptions, either on the Ising Hamiltonian or on the move. This generality is necessary for the dense Sherrington--Kirkpatrick model considered here, where the interaction graph is complete and proposal moves such as the uniform and quantum-enhanced moves may flip an extensive number of spins.

This should be contrasted with the circuit construction of [Lemieux et al.](https://arxiv.org/pdf/1910.01659), which is optimized for a sparse $(k,d)$-local Ising model,
$$
E(x)=\sum_{\ell} J_{\ell}\prod_{s\in\Omega_{\ell}}x_s,
\qquad |\Omega_{\ell}|\le k,
$$
where each spin participates in at most $d$ interactions. In the two-local case, which is our case of interest, this becomes
$$
E(x)=\sum_i h_i x_i+\sum_{(i,j)\in \mathcal E}J_{ij}x_i x_j,
\qquad \deg(i)\le d.
$$

For a move $z_j$, their Boltzmann arithmetic only depends on the subset $N_j$ of spins appearing in Hamiltonian terms affected by that move. For single-spin flips they obtain $|N_j|\le kd$, while for multi-spin flips $|N_j|\le |z_j|kd$. Hence, when $k$, $d$, and $|z_j|$ are constant, the lookup-style dependence $2^{|N_j|}$ is only a constant overhead.

In our case:

* The SK model violates the bounded-degree assumption: it has $k=2$, but $d=n-1$.
* For non-local moves such as the uniform and quantum-enhanced proposals, which typically flip $\Theta(n)$ spins, the affected neighborhood is generically the whole system, so $|N_j|=\Theta(n)$. The lookup-based Boltzmann arithmetic of Lemieux et al. would therefore contain a factor $2^{|N_j|}=2^{\Theta(n)}$.